In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'BBVA', '2206.pdf')

In [2]:
from document_processing import DocumentProcessingFacade
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

doc_processor = DocumentProcessingFacade(statement_path)
statement_properties = doc_processor.get_statement_properties()
bank = statement_properties['bank']
statement_type = statement_properties['statement_type']

print(bank, ' - ', statement_type)

bbva  -  debit


In [3]:
corrected_extracted_words = doc_processor.get_corrected_extracted_words()
corrected_extracted_words

,page,text,x0,top,x1,bottom
0,1,Estado,510.546000,1.219000,544.314000,13.219000
1,1,de,546.858000,1.219000,559.638000,13.219000
2,1,Cuenta,562.182000,1.219000,598.182000,13.219000
3,1,Libretón,459.986607,16.654410,498.904607,27.654410
4,1,Básico,501.236607,16.654410,530.397608,27.654410
...,...,...,...,...,...,...
2006,8,de,500.295960,773.127129,508.503960,782.127129
2007,8,la,510.555960,773.127129,516.297960,782.127129
2008,8,Ley,518.349960,773.127129,530.247960,782.127129
2009,8,Personas,532.299960,773.127129,563.475960,782.127129


In [4]:
from table_processing import TableProcessingFacade

table_processor = TableProcessingFacade(corrected_extracted_words, statement_properties)
reconstructed_table = table_processor.reconstruct_table()
reconstructed_table

,Date,Description,CARGOS,ABONOS,OPERACION,LIQUIDACION
0,15/MAY,PAGO CUENTA DE TERCERO Referencia 1634723960 ...,40.00,,,
1,15/MAY,SPEI ENVIADO SANTANDER Referencia 0059269420 ...,25.00,,,
2,15/MAY,PAGO CUENTA DE TERCERO Referencia 1641104987 ...,60.00,,"266,326.99","266,451.99"
3,17/MAY,SPEI ENVIADO SANTANDER Referencia 0063053690 ...,850.00,,"265,476.99","265,476.99"
4,18/MAY,SPEI ENVIADO STP Referencia 0064298989 646 18...,"80,000.00",,,
5,18/MAY,SPEI ENVIADO STP Referencia 0064318415 646 18...,500.00,,,
6,18/MAY,SAT Referencia GUIA:0095194 REF:042229R345003...,"13,621.00",,,
7,18/MAY,SAT Referencia GUIA:0112519 REF:042229QU05003...,"7,398.00",,,
8,18/MAY,SAT Referencia GUIA:0132561 REF:042229QD92003...,"3,832.00",,,
9,18/MAY,SAT Referencia GUIA:0146586 REF:042229Q995003...,"4,971.00",,,


In [5]:
from data_processing import DataProcessingFacade

data_processor = DataProcessingFacade(corrected_extracted_words, reconstructed_table, statement_properties)
df_transactions = data_processor.get_normalized_table()
df_transactions

,Date,Description,Amount,Type
0,2022-05-15,PAGO CUENTA DE TERCERO Referencia 1634723960 ...,-40.00,Cargo
2,2022-05-15,PAGO CUENTA DE TERCERO Referencia 1641104987 ...,-60.00,Cargo
36,2022-05-15,Saldo Inicial,266451.99,Saldo Inicial
1,2022-05-15,SPEI ENVIADO SANTANDER Referencia 0059269420 ...,-25.00,Cargo
3,2022-05-17,SPEI ENVIADO SANTANDER Referencia 0063053690 ...,-850.00,Cargo
4,2022-05-18,SPEI ENVIADO STP Referencia 0064298989 646 18...,-80000.00,Cargo
5,2022-05-18,SPEI ENVIADO STP Referencia 0064318415 646 18...,-500.00,Cargo
6,2022-05-18,SAT Referencia GUIA:0095194 REF:042229R345003...,-13621.00,Cargo
7,2022-05-18,SAT Referencia GUIA:0112519 REF:042229QU05003...,-7398.00,Cargo
8,2022-05-18,SAT Referencia GUIA:0132561 REF:042229QD92003...,-3832.00,Cargo


In [7]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_transactions)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_data(file_path)